In [ ]:
import time
import logging
from pathlib import Path
import sys
import json
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

logging.getLogger("transformers").setLevel(logging.ERROR)

sys.path.append("../../utils/")
from utils import *

In [ ]:
k_inicio = 1
n_folds = 5

dataset_name = "pd_CIC17__NearMiss_SMOTE__v1"

batch_size_pred = 16
source_max_token_len = 150
target_max_token_len = 3
use_gpu = False

In [ ]:
BASE_DIR = Path("../../..").resolve()

ruta_base_dataset = BASE_DIR / "02_datasets" / "processed" / dataset_name

output_base_path = (
    BASE_DIR
    / "04_experimentos"
    / "modelos"
    / f"{dataset_name}__outputs"
)

ruta_test = output_base_path / "test_final_all_folds"
ruta_test.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("Ruta dataset:", ruta_base_dataset)
print("Ruta modelos:", output_base_path)
print("Ruta salida test:", ruta_test)

In [ ]:
# =========================
# CARGA TEST
# =========================

nombre_test = f"{dataset_name}__test.csv"

df_test = cargar_dataset(nombre_test, ruta_base_dataset)

df_test["source_text"] = df_test["source_text"].astype(str)
df_test["target_text"] = df_test["target_text"].astype(str)

test_texts = df_test["source_text"].tolist()
y_true = df_test["target_text"].tolist()

valid_labels = sorted(set(y_true), key=lambda x: int(x))

print("Test shape:", df_test.shape)
print("Labels:", valid_labels)
df_test.head()

In [ ]:
def evaluar_fold(k):
    print(f"\n==============================")
    print(f" EVALUANDO FOLD {k}")
    print(f"==============================")

    output_path = output_base_path / f"{dataset_name}__outputs__{k}_{n_folds}"

    print("Modelo:", output_path)

    model = SimpleT5Wrapper()
    model.from_pretrained("t5", str(output_path), use_gpu=use_gpu)

    inicio = time.time()

    y_pred = model.predict(
        test_texts,
        batch_size=batch_size_pred,
        source_max_token_len=source_max_token_len,
        target_max_token_len=target_max_token_len
    )

    fin = time.time()
    tiempo_min = (fin - inicio) / 60

    y_pred_str = [str(p).strip() for p in y_pred]
    y_pred_final = [limpiar_prediccion(p, valid_labels) for p in y_pred_str]

    n_invalid = sum(p == "INVALID" for p in y_pred_final)

    print("Predicciones:", len(y_pred_final))
    print("Inválidas:", n_invalid)
    print(f"% inválidas: {n_invalid / len(y_pred_final):.4%}")
    print(f"Tiempo predicción: {tiempo_min:.2f} min")

    accuracy = accuracy_score(y_true, y_pred_final)

    precision_macro = precision_score(
        y_true, y_pred_final, average="macro", zero_division=0
    )
    recall_macro = recall_score(
        y_true, y_pred_final, average="macro", zero_division=0
    )
    f1_macro = f1_score(
        y_true, y_pred_final, average="macro", zero_division=0
    )

    precision_weighted = precision_score(
        y_true, y_pred_final, average="weighted", zero_division=0
    )
    recall_weighted = recall_score(
        y_true, y_pred_final, average="weighted", zero_division=0
    )
    f1_weighted = f1_score(
        y_true, y_pred_final, average="weighted", zero_division=0
    )

    mcc = matthews_corrcoef(y_true, y_pred_final)

    metricas = {
        "fold": k,
        "modelo": output_path.name,
        "accuracy": accuracy,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
        "mcc": mcc,
        "tiempo_min": tiempo_min,
        "n_invalid": n_invalid,
        "pct_invalid": n_invalid / len(y_pred_final)
    }

    df_metricas_fold = pd.DataFrame([metricas])
    df_metricas_fold.to_csv(
        ruta_test / f"metricas_test_fold_{k}.csv",
        index=False
    )

    df_preds = pd.DataFrame({
        "source_text": test_texts,
        "target_text": y_true,
        "prediction_raw": y_pred_str,
        "prediction": y_pred_final
    })

    df_preds["correct"] = df_preds["target_text"] == df_preds["prediction"]

    df_preds.to_csv(
        ruta_test / f"predicciones_test_fold_{k}.csv",
        index=False
    )

    report_dict = classification_report(
        y_true,
        y_pred_final,
        output_dict=True,
        zero_division=0
    )

    df_report = pd.DataFrame(report_dict).transpose()
    df_report.to_csv(
        ruta_test / f"classification_report_test_fold_{k}.csv"
    )

    labels_sorted = valid_labels.copy()

    if "INVALID" in y_pred_final:
        labels_sorted = labels_sorted + ["INVALID"]

    cm = confusion_matrix(
        y_true,
        y_pred_final,
        labels=labels_sorted
    )

    df_cm = pd.DataFrame(
        cm,
        index=labels_sorted,
        columns=labels_sorted
    )

    df_cm.to_csv(
        ruta_test / f"confusion_matrix_test_fold_{k}.csv"
    )

    cm_normalized = confusion_matrix(
        y_true,
        y_pred_final,
        labels=labels_sorted,
        normalize="true"
    )

    df_cm_normalized = pd.DataFrame(
        cm_normalized,
        index=labels_sorted,
        columns=labels_sorted
    )

    df_cm_normalized.to_csv(
        ruta_test / f"confusion_matrix_normalized_test_fold_{k}.csv"
    )

    fig, ax = plt.subplots(figsize=(10, 8))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=labels_sorted
    )
    disp.plot(ax=ax, cmap="Blues", xticks_rotation=90, colorbar=False)
    plt.title(f"Matriz de confusión - Fold {k}")
    plt.tight_layout()
    plt.savefig(ruta_test / f"matriz_confusion_fold_{k}.png", dpi=300)
    plt.close()

    fig, ax = plt.subplots(figsize=(10, 8))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm_normalized,
        display_labels=labels_sorted
    )
    disp.plot(
        ax=ax,
        cmap="Blues",
        xticks_rotation=90,
        values_format=".2f",
        colorbar=False
    )
    plt.title(f"Matriz de confusión normalizada - Fold {k}")
    plt.tight_layout()
    plt.savefig(
        ruta_test / f"matriz_confusion_normalizada_fold_{k}.png",
        dpi=300
    )
    plt.close()

    print("\nMétricas fold:")
    for key, value in metricas.items():
        if isinstance(value, float):
            print(f"{key}: {value:.6f}")
        else:
            print(f"{key}: {value}")

    return metricas

In [ ]:
all_metrics = []

for k in range(k_inicio, n_folds + 1):
    metricas_fold = evaluar_fold(k)
    all_metrics.append(metricas_fold)

df_all_metrics = pd.DataFrame(all_metrics)

df_all_metrics.to_csv(
    ruta_test / "metricas_test_all_folds.csv",
    index=False
)

df_all_metrics

In [ ]:
metric_cols = [
    "accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "precision_weighted",
    "recall_weighted",
    "f1_weighted",
    "mcc",
    "tiempo_min",
    "n_invalid",
    "pct_invalid"
]

df_mean = df_all_metrics[metric_cols].mean()
df_var = df_all_metrics[metric_cols].var()
df_std = df_all_metrics[metric_cols].std()

df_resumen = pd.DataFrame({
    "mean": df_mean,
    "var": df_var,
    "std": df_std
})

df_resumen.to_csv(
    ruta_test / "metricas_test_resumen_mean_var_std.csv"
)

df_resumen

In [ ]:
print("\n===== RESUMEN FINAL TEST ALL FOLDS =====")

for metric in metric_cols:
    print(
        f"{metric:20s}: "
        f"{df_resumen.loc[metric, 'mean']:.6f} ± "
        f"{df_resumen.loc[metric, 'std']:.6f}"
    )

In [ ]:
resumen_dict = {
    "dataset_name": dataset_name,
    "n_folds": n_folds,
    "metricas_por_fold": df_all_metrics.to_dict(orient="records"),
    "mean": df_mean.to_dict(),
    "var": df_var.to_dict(),
    "std": df_std.to_dict()
}

with open(
    ruta_test / "metricas_test_resumen_all_folds.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(resumen_dict, f, indent=4)

print("Resumen guardado en:", ruta_test)